# Lab 01 · How a Spark application runs

**Part A** (about 45 min): watch Spark run your code, in the Spark UI and in query plans.<br>
**Part B** (about 75 min): 15 short DataFrame problems: select, filter, new columns, sorting, missing values, text and dates.

**Checklist questions Part A gives you evidence for:**<br>
<span style="color:#d93025"><b>1.1</b> Apache Spark architecture · <b>1.2</b> What is a SparkSession? · <b>1.3</b> Driver vs. executor · <b>1.5</b> Narrow vs. wide dependencies · <b>1.7</b> Fault tolerance · <b>1.10</b> Lifecycle of a Spark job · <b>1.11</b> Transformation vs. action · <b>1.12</b> Lazy evaluation</span>

## How to work
* Run the **Setup** cell first, then answer the questions in order. `F` is `pyspark.sql.functions`.
* **Part A**: run the code under the question (or add a line), look at the output or the Spark UI, and write your answer as a comment.
* **Part B**: write the code yourself. Your result must look exactly like the **expected output**, with the rows in the same order.
* The answers are collapsed at the very end of the notebook. Open one only after you have tried.

## Documentation
* [Cluster Mode Overview](https://spark.apache.org/docs/latest/cluster-overview.html): driver, executors, tasks
* [Master URLs](https://spark.apache.org/docs/latest/submitting-applications.html#master-urls): what `local[4]` means
* [RDD operations](https://spark.apache.org/docs/latest/rdd-programming-guide.html#rdd-operations): transformations, actions, lazy evaluation
* [Web UI](https://spark.apache.org/docs/latest/web-ui.html): the Jobs, Stages and Executors tabs
* [Tuning partitions](https://spark.apache.org/docs/latest/sql-performance-tuning.html#tuning-partitions): `spark.sql.shuffle.partitions`
* [DataFrame quickstart](https://spark.apache.org/docs/latest/api/python/getting_started/quickstart_df.html): select, filter and withColumn in examples
* [DataFrame methods](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html): `filter`, `select`, `withColumn`, `orderBy`, `distinct`, `na.fill`
* [Column methods](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/column.html): `isin`, `between`, `isNull`, `alias`, `desc`
* [Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html): `when`, `round`, `split`, `concat`, `coalesce`, `date_format`, `datediff`
* [Datetime patterns](https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html): the letters for `date_format`, like `yyyy-MM`

## The data
Three small tables of an online shop. The setup cell creates them, and you can read every row in it.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date* (the day the customer registered)
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)

Good to know:
* order 1011 is a guest checkout: it has no *customer_id*
* customer 6 has no email and no city
* customer 7 has never ordered, and product 107 has never been sold
* order 1016 is for product 108, which is no longer in *products*

## Setup
Run this cell once. It starts Spark, saves the three tables as Parquet files in `data/shop/` and loads them into `customers`, `products` and `orders`.

In [ ]:
# Setup: starts Spark and creates the three shop tables. Run it once.
import logging, os, shutil, sys, time
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[4,2]")   # question A1 asks what the numbers mean
         .appName("lab01")
         .config("spark.sql.adaptive.enabled", "false")       # AQE stays off until Lab 03, so plans stay simple
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")                  # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers_rows = [
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
]
products_rows = [
    # product_id, product_name, category, price
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
]
orders_rows = [
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
]

# Write the tables as Parquet files and read them back, the way real data arrives.
# data/shop belongs to this lab: it is rebuilt every time you run this cell.
shutil.rmtree("data/shop", ignore_errors=True)
for table, rows, schema in [
    ("customers", customers_rows, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    ("products",  products_rows,  "product_id INT, product_name STRING, category STRING, price DOUBLE"),
    ("orders",    orders_rows,    "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING"),
]:
    spark.createDataFrame(rows, schema).coalesce(1).write.parquet(f"data/shop/{table}")

customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")

def count_jobs(action):
    """Runs action() and returns how many Spark jobs it started (question A5)."""
    tracker = spark.sparkContext.statusTracker()
    before = set(tracker.getJobIdsForGroup())
    action()
    time.sleep(0.5)                       # the Spark UI records a job a moment after it ends
    return len(set(tracker.getJobIdsForGroup()) - before)

print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

Look at the three tables:

In [ ]:
customers.show()
products.show()
orders.show()

## Part A · How Spark runs your code
Keep the Spark UI open in your browser: the setup cell prints its address (port 4040, or 4041 if 4040 is taken).

### Driver, executors and the SparkSession <span style="color:#d93025">(1.1, 1.2, 1.3)</span>

**A1. What do the two numbers in `local[4,2]` mean? The setup cell started Spark with this master.**

In [ ]:
# My answer:
spark.sparkContext.master

**A2. Open the Spark UI → *Executors* tab. How many executors are there, and how many cores do they have?**

In [ ]:
# My answer:

**A3. Does the code inside a UDF run in your notebook's process? Run the cell and compare the two process ids.**

In [ ]:
# My answer:
import os
where = F.udf(lambda x: os.getpid(), "int")    # returns the id of the process it runs in
print("notebook (driver):", os.getpid())
print("UDF:              ", orders.select(where("order_id")).first()[0])

### Transformations, actions and lazy evaluation <span style="color:#d93025">(1.11, 1.12)</span>

**A4. Line 1 divides by zero, but the error only appears at line 2. Why?**

In [ ]:
# My answer:
bad = orders.withColumn("half", F.col("quantity") / 0)   # line 1
bad.show()                                              # line 2

**A5. Which of these start a Spark job: `filter`, `select`, `groupBy("status").count()`, `printSchema()`, `count()`, `show()`, `collect()`? Guess first, then measure each one with `count_jobs` (example in the cell).**

In [ ]:
# My guess:
count_jobs(lambda: orders.filter(F.col("quantity") > 1))

### Narrow vs. wide, stages and tasks <span style="color:#d93025">(1.5, 1.10)</span>

**A6. Which of the two queries needs a shuffle? Look for `Exchange` in their `explain()` output.**

In [ ]:
# My answer:
q1 = orders.filter(F.col("status") == "delivered").select("order_id", "quantity")
q2 = orders.groupBy("status").count()

**A7. Run `q2.collect()`. In the Spark UI, open the newest job (top row of the Jobs tab). How many stages does it have, and how many tasks does each stage have?**

In [ ]:
# My answer:
q2.collect()

**A8. Stage 2 had 200 tasks for only 3 statuses. Set `spark.sql.shuffle.partitions` to 4, build `q2` again and collect it. How many tasks does stage 2 have now?**

In [ ]:
# My answer:
spark.conf.set("spark.sql.shuffle.partitions", 4)

### Fault tolerance <span style="color:#d93025">(1.7)</span>

**A9. The function below crashes the first time it sees order 1007, like a machine that dies. Does the job still finish? In the Spark UI, open this job's last stage: how many tasks failed?**

In [ ]:
# My answer:
import tempfile
crash_flag = os.path.join(tempfile.gettempdir(), "lab01_crashed_once")
if os.path.exists(crash_flag):
    os.remove(crash_flag)

def crash_once(order_id):
    if order_id == 1007 and not os.path.exists(crash_flag):
        open(crash_flag, "w").close()            # remember that it crashed once
        raise RuntimeError("simulated crash")
    return order_id

crash_once_udf = F.udf(crash_once, "int")
rows = orders.repartition(4).select(crash_once_udf("order_id")).collect()   # 4 partitions = 4 tasks
print("job finished with", len(rows), "rows")

**A10. Now the function crashes every time. How many times does Spark try the task before it gives up? In the Spark UI, open the failed stage (Stages tab → *Failed Stages*) and look at its *Tasks* table.**

In [ ]:
# My answer:
always_crash_udf = F.udf(lambda order_id: 1 // 0 if order_id == 1007 else order_id, "int")
orders.repartition(4).select(always_crash_udf("order_id")).collect()

**Done with Part A?** Explain each red checklist question at the top aloud, in your own words, using what you saw.

## Part B · DataFrame basics
15 short problems on the three tables. Write each one from scratch and print it with `.show()`. You are done when your table looks exactly like the expected output.

**B1. Show the name and email of the customers who live in Tashkent.**

Expected output:

```
+----------------+---------------+
|            name|          email|
+----------------+---------------+
|    Aziz Karimov|   aziz@mail.uz|
|Dilnoza Rahimova|dilnoza@mail.uz|
+----------------+---------------+
```

<details><summary>Hint</summary>

`filter` keeps rows, `select` keeps columns. A condition looks like `F.col("city") == "Tashkent"`.

</details>

**B2. Show the orders with a quantity of 3 or more: order_id, product_id and quantity, biggest quantity first.**

Expected output:

```
+--------+----------+--------+
|order_id|product_id|quantity|
+--------+----------+--------+
|    1014|       105|       5|
|    1009|       105|       4|
|    1002|       105|       3|
+--------+----------+--------+
```

<details><summary>Hint</summary>

`orderBy(F.col("quantity").desc())`

</details>

**B3. Add the column `price_with_vat`: the price plus 12% VAT, rounded to 2 decimals. Show product_name, price and price_with_vat.**

Expected output:

```
+------------+------+--------------+
|product_name| price|price_with_vat|
+------------+------+--------------+
|      Laptop|1200.0|        1344.0|
|  Headphones| 150.0|         168.0|
|Coffee Maker|  85.0|          95.2|
|     Blender|  60.0|          67.2|
|       Novel|  15.0|          16.8|
|    Cookbook|  25.0|          28.0|
|   Desk Lamp|  40.0|          44.8|
+------------+------+--------------+
```

<details><summary>Hint</summary>

`withColumn(name, expression)` adds a column. `F.round(column, 2)` rounds.

</details>

**B4. Sort the products by category from A to Z, and inside each category by price from high to low. Show category, product_name and price.**

Expected output:

```
+-----------+------------+------+
|   category|product_name| price|
+-----------+------------+------+
|      Books|    Cookbook|  25.0|
|      Books|       Novel|  15.0|
|Electronics|      Laptop|1200.0|
|Electronics|  Headphones| 150.0|
|       Home|   Desk Lamp|  40.0|
|    Kitchen|Coffee Maker|  85.0|
|    Kitchen|     Blender|  60.0|
+-----------+------------+------+
```

<details><summary>Hint</summary>

`orderBy` takes several columns, each with its own `.asc()` or `.desc()`.

</details>

**B5. Give every product a `price_level`: `cheap` if the price is under 50, `mid` from 50 to 200, `premium` above 200. Show product_name, price and price_level.**

Expected output:

```
+------------+------+-----------+
|product_name| price|price_level|
+------------+------+-----------+
|      Laptop|1200.0|    premium|
|  Headphones| 150.0|        mid|
|Coffee Maker|  85.0|        mid|
|     Blender|  60.0|        mid|
|       Novel|  15.0|      cheap|
|    Cookbook|  25.0|      cheap|
|   Desk Lamp|  40.0|      cheap|
+------------+------+-----------+
```

<details><summary>Hint</summary>

`F.when(condition, value).when(condition, value).otherwise(value)`: the first true condition wins.

</details>

**B6. Show the orders placed in February 2024: order_id and order_date, oldest first.**

Expected output:

```
+--------+----------+
|order_id|order_date|
+--------+----------+
|    1004|2024-02-03|
|    1005|2024-02-05|
|    1006|2024-02-15|
|    1007|2024-02-20|
+--------+----------+
```

<details><summary>Hint</summary>

`Column.between(low, high)` (both ends included), or `F.year(...)` and `F.month(...)`.

</details>

**B7. Show the orders that are `shipped` or `cancelled` and have a quantity of 2 or more: order_id, status and quantity.**

Expected output:

```
+--------+---------+--------+
|order_id|   status|quantity|
+--------+---------+--------+
|    1006|  shipped|       2|
|    1010|cancelled|       2|
|    1012|  shipped|       2|
+--------+---------+--------+
```

<details><summary>Hint</summary>

`Column.isin(...)`. Combine conditions with `&` (and) or `|` (or), and put each comparison in parentheses.

</details>

**B8. Which customers have no email? Show customer_id and name.**

Expected output:

```
+-----------+------------+
|customer_id|        name|
+-----------+------------+
|          6|Farid Aliyev|
+-----------+------------+
```

<details><summary>Hint</summary>

`Column.isNull()`. `F.col("email") == None` does not work: a comparison with null is never true.

</details>

**B9. Show customer_id, name and city of every customer, with `Unknown` where the city is missing. Call the name column `customer_name`.**

Expected output:

```
+-----------+----------------+--------+
|customer_id|   customer_name|    city|
+-----------+----------------+--------+
|          1|    Aziz Karimov|Tashkent|
|          2|   Bella Schmidt|  Berlin|
|          3|     Carlos Ruiz|  Madrid|
|          4|Dilnoza Rahimova|Tashkent|
|          5|     Emma Wilson|  London|
|          6|    Farid Aliyev| Unknown|
|          7|       Grace Lee|  Berlin|
+-----------+----------------+--------+
```

<details><summary>Hint</summary>

`withColumnRenamed(old, new)`. `F.coalesce(a, b)` returns the first value that is not null; or use `na.fill`.

</details>

**B10. Split `name` into `first_name` and `last_name`. Show customer_id, first_name and last_name.**

Expected output:

```
+-----------+----------+---------+
|customer_id|first_name|last_name|
+-----------+----------+---------+
|          1|      Aziz|  Karimov|
|          2|     Bella|  Schmidt|
|          3|    Carlos|     Ruiz|
|          4|   Dilnoza| Rahimova|
|          5|      Emma|   Wilson|
|          6|     Farid|   Aliyev|
|          7|     Grace|      Lee|
+-----------+----------+---------+
```

<details><summary>Hint</summary>

`F.split(column, " ")` returns a list; `[0]` takes its first item.

</details>

**B11. Show the email domain (the part after `@`) of every customer who has an email. Show name and domain.**

Expected output:

```
+----------------+----------+
|            name|    domain|
+----------------+----------+
|    Aziz Karimov|   mail.uz|
|   Bella Schmidt|    web.de|
|     Carlos Ruiz| correo.es|
|Dilnoza Rahimova|   mail.uz|
|     Emma Wilson|post.co.uk|
|       Grace Lee|    web.de|
+----------------+----------+
```

<details><summary>Hint</summary>

`F.split` again, or `F.substring_index`.

</details>

**B12. Which cities do the customers live in? One row per city, no empty values, from A to Z.**

Expected output:

```
+--------+
|    city|
+--------+
|  Berlin|
|  London|
|  Madrid|
|Tashkent|
+--------+
```

<details><summary>Hint</summary>

`dropna()` removes rows with nulls, `distinct()` removes repeated rows.

</details>

**B13. Take the first 5 orders by order_id. Show order_id, order_date, the month as `order_month` (like `2024-01`) and the day of the week as `weekday` (like `Wed`).**

Expected output:

```
+--------+----------+-----------+-------+
|order_id|order_date|order_month|weekday|
+--------+----------+-----------+-------+
|    1001|2024-01-10|    2024-01|    Wed|
|    1002|2024-01-10|    2024-01|    Wed|
|    1003|2024-01-25|    2024-01|    Thu|
|    1004|2024-02-03|    2024-02|    Sat|
|    1005|2024-02-05|    2024-02|    Mon|
+--------+----------+-----------+-------+
```

<details><summary>Hint</summary>

`F.date_format(column, pattern)` with the patterns `yyyy-MM` and `E`.

</details>

**B14. How many days had each customer been registered on 2024-04-01? Show name, signup_date and days_registered.**

Expected output:

```
+----------------+-----------+---------------+
|            name|signup_date|days_registered|
+----------------+-----------+---------------+
|    Aziz Karimov| 2024-01-05|             87|
|   Bella Schmidt| 2024-01-20|             72|
|     Carlos Ruiz| 2024-02-02|             59|
|Dilnoza Rahimova| 2024-02-14|             47|
|     Emma Wilson| 2024-03-01|             31|
|    Farid Aliyev| 2024-03-10|             22|
|       Grace Lee| 2024-03-15|             17|
+----------------+-----------+---------------+
```

<details><summary>Hint</summary>

`F.datediff(end, start)`. Make the end date with `F.to_date(F.lit("2024-04-01"))`.

</details>

**B15. Make a short name for every customer: the first name, a space, the first letter of the last name and a dot, like `Aziz K.` Show customer_id and short_name.**

Expected output:

```
+-----------+----------+
|customer_id|short_name|
+-----------+----------+
|          1|   Aziz K.|
|          2|  Bella S.|
|          3| Carlos R.|
|          4|Dilnoza R.|
|          5|   Emma W.|
|          6|  Farid A.|
|          7|  Grace L.|
+-----------+----------+
```

<details><summary>Hint</summary>

`F.concat(...)` glues strings together; `F.substring(column, 1, 1)` takes the first letter; `F.lit` makes a fixed value.

</details>

---
**Done?** In the Theory part of Day 1, answer the red checklist questions aloud.

## Answers
Try each question yourself first. Click a question to open its answer; Part A answers also show the output of the reference run.

<details>
<summary><b>A1</b> &nbsp;What do the two numbers in local[4,2] mean? The setup cell started Spark with this master.</summary>

**4**: Spark runs 4 tasks at the same time (4 threads; `defaultParallelism` is 4). **2**: a task that fails is tried at most 2 times. `local` means everything runs in one process on your machine: the driver and the executor share one JVM.

`spark` is the **SparkSession**, your entry point: it holds the settings, creates DataFrames, runs SQL, and it started the SparkContext that sends work to the executors.

```python
spark.sparkContext.master, spark.sparkContext.defaultParallelism
```

Result:

```
('local[4,2]', 4)
```

</details>

<details>
<summary><b>A2</b> &nbsp;Open the Spark UI → Executors tab. How many executors are there, and how many cores do they have?</summary>

**One row, `driver`, with 4 cores.** In local mode the driver also does the executors' work. On a cluster you would see the driver plus one row per executor, each with its own cores and memory: the **driver** plans the work and hands out tasks, the **executors** run the tasks and hold data. A cluster manager (YARN, Kubernetes, standalone) starts the executors for you.

</details>

<details>
<summary><b>A3</b> &nbsp;Does the code inside a UDF run in your notebook's process? Run the cell and compare the two process ids.</summary>

**No, the two ids differ.** Your notebook is the **driver**: it builds the plan and sends tasks out. The tasks run in an **executor**, and Python code inside them runs in separate Python worker processes. On a cluster these are on other machines; only what you ask for (`show`, `collect`) comes back to the driver.

```python
import os
where = F.udf(lambda x: os.getpid(), "int")    # returns the id of the process it runs in
print("notebook (driver):", os.getpid())
print("UDF:              ", orders.select(where("order_id")).first()[0])
```

Result:

```
notebook (driver): 1055
UDF:               1152
```

*Your process ids will be different.*

</details>

<details>
<summary><b>A4</b> &nbsp;Line 1 divides by zero, but the error only appears at line 2. Why?</summary>

`withColumn` is a **transformation**: Spark only adds a step to the plan. It checks names and types, but it reads no data. `show` is an **action**: now Spark runs a job, computes the column and hits the division by zero. This is **lazy evaluation**: nothing runs until an action needs a result, so a bug in your data logic shows up at the first action, not on the line where you wrote it.

```python
bad = orders.withColumn("half", F.col("quantity") / 0)   # line 1
bad.show()                                              # line 2
```

Result:

```
ArithmeticException: [DIVIDE_BY_ZERO] Division by zero. Use `try_divide` to tolerate divisor being 0 and return NULL instead. If necessary set "spark.sql.ansi.enabled" to "false" to bypass this error. SQLSTATE: 22012
```

</details>

<details>
<summary><b>A5</b> &nbsp;Which of these start a Spark job: filter, select, groupBy("status").count(), printSchema(), count(), show(), ...</summary>

**Only `count()`, `show()` and `collect()`**: they are **actions**, they give a result back to you, so Spark must run a job. `filter`, `select` and `groupBy(...).count()` are **transformations**: they return a new DataFrame and run nothing. Careful: `count()` after `groupBy` is a transformation, `count()` on a DataFrame is an action. `printSchema()` needs no job either: the schema is stored in the Parquet file.

```python
for name, action in [
    ("filter", lambda: orders.filter(F.col("quantity") > 1)),
    ("select", lambda: orders.select("order_id")),
    ('groupBy("status").count()', lambda: orders.groupBy("status").count()),
    ("printSchema()", lambda: orders.printSchema()),
    ("count()", lambda: orders.count()),
    ("show()", lambda: orders.show(3)),
    ("collect()", lambda: orders.collect()),
]:
    print(f"{name:27} -> {count_jobs(action)} job(s)")
```

Result:

```
filter                      -> 0 job(s)
select                      -> 0 job(s)
groupBy("status").count()   -> 0 job(s)
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)

printSchema()               -> 0 job(s)
count()                     -> 1 job(s)
+--------+-----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|quantity|order_date|   status|
+--------+-----------+----------+--------+----------+---------+
|    1001|          1|       101|       1|2024-01-10|delivered|
|    1002|          1|       105|       3|2024-01-10|delivered|
|    1003|          2|       102|       2|2024-01-25|delivered|
+--------+-----------+----------+--------+----------+---------+
only showing top 3 rows
show()                      -> 1 job(s)
collect()                   -> 1 job(s)
```

</details>

<details>
<summary><b>A6</b> &nbsp;Which of the two queries needs a shuffle? Look for Exchange in their explain() output.</summary>

**`q2`.** Its plan has `Exchange hashpartitioning(status, 200)`: rows with the same status must end up in the same partition, so Spark moves rows between partitions. That is a **shuffle**, and it makes `groupBy` a **wide** transformation. `q1` has no `Exchange`: `filter` and `select` work on each partition alone, so they are **narrow**.

```python
q1 = orders.filter(F.col("status") == "delivered").select("order_id", "quantity")
q2 = orders.groupBy("status").count()
q1.explain()
q2.explain()
```

Result:

```
== Physical Plan ==
*(1) Project [order_id#24, quantity#27]
+- *(1) Filter (isnotnull(status#29) AND (status#29 = delivered))
   +- *(1) ColumnarToRow
      +- FileScan parquet [order_id#24,quantity#27,status#29] Batched: true, DataFilters: [isnotnull(status#29), (status#29 = delivered)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [IsNotNull(status), EqualTo(status,delivered)], ReadSchema: struct<order_id:int,quantity:int,status:string>


== Physical Plan ==
*(2) HashAggregate(keys=[status#29], functions=[count(1)])
+- Exchange hashpartitioning(status#29, 200), ENSURE_REQUIREMENTS, [plan_id=244]
   +- *(1) HashAggregate(keys=[status#29], functions=[partial_count(1)])
      +- *(1) ColumnarToRow
         +- FileScan parquet [status#29] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<status:string>
```

</details>

<details>
<summary><b>A7</b> &nbsp;Run q2.collect(). In the Spark UI, open the newest job (top row of the Jobs tab). How many stages does it ...</summary>

**2 stages: the first has 1 task, the second has 200.** The job is cut into stages at the shuffle. Stage 1 reads the file and counts the rows of each status in its partition: 1 task, because `orders` is one small file, so 1 partition (`orders.rdd.getNumPartitions()` returns 1). Stage 2 reads the shuffled rows and adds up the counts: 200 tasks.

This is the **lifecycle of a job**: an action starts a job → the job is cut into stages at every shuffle → each stage runs one task per partition on the executors → the result goes back to the driver.

```python
print(q2.collect())
print(orders.rdd.getNumPartitions())
```

Result:

```
[Row(status='shipped', count=4), Row(status='cancelled', count=2), Row(status='delivered', count=10)]
1
```

</details>

<details>
<summary><b>A8</b> &nbsp;Stage 2 had 200 tasks for only 3 statuses. Set spark.sql.shuffle.partitions to 4, build q2 again and collect ...</summary>

**4.** After a shuffle, Spark makes `spark.sql.shuffle.partitions` partitions (default 200), and each partition is one task, even an empty one: most of the 200 tasks had nothing to do. You have to build `q2` again because a DataFrame keeps the plan it made at its first action. In Lab 03, AQE picks this number for you.

```python
spark.conf.set("spark.sql.shuffle.partitions", 4)
q2 = orders.groupBy("status").count()
q2.collect()
```

Result:

```
[Row(status='shipped', count=4),
 Row(status='delivered', count=10),
 Row(status='cancelled', count=2)]
```

</details>

<details>
<summary><b>A9</b> &nbsp;The function below crashes the first time it sees order 1007, like a machine that dies. Does the job still ...</summary>

**Yes, the job finishes** (`job finished with 16 rows`). In the last stage (4 tasks), **1 task failed**, and Spark ran only that task again; the other 3 were not repeated. Spark can redo a task because it knows the **lineage**: the steps that build each partition from the source file. When a whole executor is lost, Spark does the same for every partition that executor held.

```python
import tempfile
crash_flag = os.path.join(tempfile.gettempdir(), "lab01_crashed_once")
if os.path.exists(crash_flag):
    os.remove(crash_flag)

def crash_once(order_id):
    if order_id == 1007 and not os.path.exists(crash_flag):
        open(crash_flag, "w").close()            # remember that it crashed once
        raise RuntimeError("simulated crash")
    return order_id

crash_once_udf = F.udf(crash_once, "int")
rows = orders.repartition(4).select(crash_once_udf("order_id")).collect()   # 4 partitions = 4 tasks
print("job finished with", len(rows), "rows")
```

Result:

```
job finished with 16 rows
```

</details>

<details>
<summary><b>A10</b> &nbsp;Now the function crashes every time. How many times does Spark try the task before it gives up? In the Spark ...</summary>

**2 times**: the task that holds order 1007 has two rows in the Tasks table, attempt 0 and attempt 1, both FAILED. That is the 2 in `local[4,2]` (on a cluster it is the setting `spark.task.maxFailures`, default 4). Then Spark gives up on the whole job (its other tasks may show KILLED) and raises the last error (`ZeroDivisionError`) in your notebook. Retrying helps with temporary problems like a lost machine, not with a bug in your code.

```python
always_crash_udf = F.udf(lambda order_id: 1 // 0 if order_id == 1007 else order_id, "int")
orders.repartition(4).select(always_crash_udf("order_id")).collect()
```

Result:

```
PythonException: An exception was thrown from the Python worker:
...
ZeroDivisionError: integer division or modulo by zero
```

</details>

<details>
<summary><b>B1</b> &nbsp;Show the name and email of the customers who live in Tashkent.</summary>

```python
customers.filter(F.col("city") == "Tashkent").select("name", "email").show()
```

</details>

<details>
<summary><b>B2</b> &nbsp;Show the orders with a quantity of 3 or more: order_id, product_id and quantity, biggest quantity first.</summary>

```python
(orders.filter(F.col("quantity") >= 3)
       .select("order_id", "product_id", "quantity")
       .orderBy(F.col("quantity").desc())
       .show())
```

</details>

<details>
<summary><b>B3</b> &nbsp;Add the column price_with_vat: the price plus 12% VAT, rounded to 2 decimals. Show product_name, price and ...</summary>

```python
(products.withColumn("price_with_vat", F.round(F.col("price") * 1.12, 2))
         .select("product_name", "price", "price_with_vat")
         .show())
```

</details>

<details>
<summary><b>B4</b> &nbsp;Sort the products by category from A to Z, and inside each category by price from high to low. Show category, ...</summary>

```python
(products.select("category", "product_name", "price")
         .orderBy(F.col("category").asc(), F.col("price").desc())
         .show())
```

</details>

<details>
<summary><b>B5</b> &nbsp;Give every product a price_level: cheap if the price is under 50, mid from 50 to 200, premium above 200. Show ...</summary>

```python
(products.withColumn("price_level",
                     F.when(F.col("price") < 50, "cheap")
                      .when(F.col("price") <= 200, "mid")
                      .otherwise("premium"))
         .select("product_name", "price", "price_level")
         .show())
```

</details>

<details>
<summary><b>B6</b> &nbsp;Show the orders placed in February 2024: order_id and order_date, oldest first.</summary>

```python
(orders.filter(F.col("order_date").between("2024-02-01", "2024-02-29"))
       .select("order_id", "order_date")
       .orderBy("order_date")
       .show())
```

Also right: `.filter((F.year("order_date") == 2024) & (F.month("order_date") == 2))`.

</details>

<details>
<summary><b>B7</b> &nbsp;Show the orders that are shipped or cancelled and have a quantity of 2 or more: order_id, status and quantity.</summary>

```python
(orders.filter(F.col("status").isin("shipped", "cancelled") & (F.col("quantity") >= 2))
       .select("order_id", "status", "quantity")
       .show())
```

</details>

<details>
<summary><b>B8</b> &nbsp;Which customers have no email? Show customer_id and name.</summary>

```python
customers.filter(F.col("email").isNull()).select("customer_id", "name").show()
```

</details>

<details>
<summary><b>B9</b> &nbsp;Show customer_id, name and city of every customer, with Unknown where the city is missing. Call the name ...</summary>

```python
(customers.withColumnRenamed("name", "customer_name")
          .withColumn("city", F.coalesce(F.col("city"), F.lit("Unknown")))
          .select("customer_id", "customer_name", "city")
          .show())
```

Also right: `customers.na.fill({"city": "Unknown"})`.

</details>

<details>
<summary><b>B10</b> &nbsp;Split name into first_name and last_name. Show customer_id, first_name and last_name.</summary>

```python
parts = F.split(F.col("name"), " ")
customers.select("customer_id", parts[0].alias("first_name"), parts[1].alias("last_name")).show()
```

</details>

<details>
<summary><b>B11</b> &nbsp;Show the email domain (the part after @) of every customer who has an email. Show name and domain.</summary>

```python
(customers.filter(F.col("email").isNotNull())
          .select("name", F.split(F.col("email"), "@")[1].alias("domain"))
          .show())
```

Also right: `F.substring_index("email", "@", -1)` or `F.regexp_extract("email", "@(.+)", 1)`.

</details>

<details>
<summary><b>B12</b> &nbsp;Which cities do the customers live in? One row per city, no empty values, from A to Z.</summary>

```python
customers.select("city").dropna().distinct().orderBy("city").show()
```

</details>

<details>
<summary><b>B13</b> &nbsp;Take the first 5 orders by order_id. Show order_id, order_date, the month as order_month (like 2024-01) and ...</summary>

```python
(orders.orderBy("order_id")
       .limit(5)
       .select("order_id", "order_date",
               F.date_format("order_date", "yyyy-MM").alias("order_month"),
               F.date_format("order_date", "E").alias("weekday"))
       .show())
```

</details>

<details>
<summary><b>B14</b> &nbsp;How many days had each customer been registered on 2024-04-01? Show name, signup_date and days_registered.</summary>

```python
(customers.withColumn("days_registered", F.datediff(F.to_date(F.lit("2024-04-01")), F.col("signup_date")))
          .select("name", "signup_date", "days_registered")
          .show())
```

</details>

<details>
<summary><b>B15</b> &nbsp;Make a short name for every customer: the first name, a space, the first letter of the last name and a dot, ...</summary>

```python
parts = F.split(F.col("name"), " ")
short_name = F.concat(parts[0], F.lit(" "), F.substring(parts[1], 1, 1), F.lit("."))
customers.select("customer_id", short_name.alias("short_name")).show()
```

</details>